# Confound-control method check — model-comparison vs. confound-stratified permutation
One patient (PTYEU_task147), hippocampus/self. Tests whether the variance-partitioning
(model-comparison) results are sensitive to confound-model misspecification by comparing
against an alternative null that never fits a parametric confound model at all.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

VP_DIR  = '/scratch/aniluchavez/ConvoDATAS/VPResults'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

strat_bins = pickle.load(open(f'{VP_DIR}/stratified_perm_prototype_PTYEU_hippo_self.pkl', 'rb'))
freq_mc    = pickle.load(open(f'{VP_DIR}/freqonly_modelcomp_prototype_PTYEU_hippo_self.pkl', 'rb'))
other_conf = pickle.load(open(f'{VP_DIR}/confound_check_sweep_PTYEU_hippo_self.pkl', 'rb'))

n_m = len(strat_bins)
print('neurons:', n_m)
other_conf

In [ ]:
# ── Build unified comparison table ───────────────────────────────────────────
# frequency: fine-bin (N_BINS=100, ~84 actual bins) stratified vs matched model-comparison
freq_strat_pct = 100 * strat_bins['significant_freqstrat_b100'].mean()
freq_mc_pct    = 100 * freq_mc['significant_freqonly_mc'].mean()

# unique values / actual bins achieved at N_BINS=100 request, and whether the
# confound has real explanatory power on its own (median R²) — computed earlier
bin_info = {
    'frequency':      {'unique_vals': 336,  'actual_bins': 83,  'r2_alone': 0.077},
    'word_length':    {'unique_vals': 14,   'actual_bins': 10,  'r2_alone': None},
    'sent_position':  {'unique_vals': 52,   'actual_bins': 26,  'r2_alone': None},
    'speaking_rate':  {'unique_vals': 4247, 'actual_bins': 100, 'r2_alone': None},
    'surprisal':      {'unique_vals': 4445, 'actual_bins': 100, 'r2_alone': None},
}

rows = [{'confound': 'frequency', 'pct_sig_stratified': freq_strat_pct, 'pct_sig_modelcomp': freq_mc_pct}]
for _, r in other_conf.iterrows():
    rows.append({'confound': r['confound'], 'pct_sig_stratified': r['pct_sig_stratified'],
                 'pct_sig_modelcomp': r['pct_sig_modelcomp']})
    bin_info[r['confound']]['r2_alone'] = r['median_r2_confound_alone']

comp = pd.DataFrame(rows)
comp['unique_vals']  = comp['confound'].map(lambda c: bin_info[c]['unique_vals'])
comp['actual_bins']  = comp['confound'].map(lambda c: bin_info[c]['actual_bins'])
comp['r2_alone']     = comp['confound'].map(lambda c: bin_info[c]['r2_alone'])
comp['gap']          = (comp['pct_sig_stratified'] - comp['pct_sig_modelcomp']).abs()
comp = comp.sort_values('r2_alone')
comp

In [ ]:
# ── PLOT 1: paired bars, stratified vs model-comparison, per confound ───────
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comp))
w = 0.35

ax.bar(x - w/2, comp['pct_sig_stratified'], width=w, label='stratified (fine bins)', color='#4393c3')
ax.bar(x + w/2, comp['pct_sig_modelcomp'],  width=w, label='model-comparison (xcirc)', color='#d6604d')

for i, (_, r) in enumerate(comp.iterrows()):
    ax.text(i, max(r['pct_sig_stratified'], r['pct_sig_modelcomp']) + 2,
            f"bins: {r['actual_bins']}/100\n$R^2$alone: {r['r2_alone']:.3f}",
            ha='center', fontsize=8, color='gray')

ax.set_xticks(x)
ax.set_xticklabels(comp['confound'], rotation=20, ha='right')
ax.set_ylabel('% significant neurons (hippocampus/self)')
ax.set_title('Stratified-permutation vs model-comparison, by confound')
ax.legend(frameon=False)
ax.set_ylim(0, 85)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_confound_method_comparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 2: bin-granularity sweep for frequency — convergence story ─────────
bin_sweep = [10, 20, 30, 50, 100]
pct_strat_by_bin = [100 * strat_bins[f'significant_freqstrat_b{nb}'].mean() for nb in bin_sweep]

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(bin_sweep, pct_strat_by_bin, marker='o', color='#4393c3', linewidth=2,
        markersize=8, label='stratified permutation')
ax.axhline(freq_mc_pct, color='#d6604d', linewidth=2, linestyle='--',
           label=f'model-comparison ({freq_mc_pct:.1f}%)')
ax.set_xlabel('N_BINS requested (frequency)')
ax.set_ylabel('% significant neurons')
ax.set_title('Frequency: stratified-null result converges to\nmodel-comparison as bins get finer')
ax.legend(frameon=False)
ax.set_ylim(0, 50)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_freq_bin_convergence.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 3: agreement gap vs. bin coarseness — explains the outlier ─────────
comp['bin_coarseness'] = 100 - comp['actual_bins']  # higher = coarser

fig, ax = plt.subplots(figsize=(6.5, 5))
sizes = 200 * (comp['r2_alone'] - comp['r2_alone'].min()) / (comp['r2_alone'].max() - comp['r2_alone'].min() + 1e-6) + 40
sc = ax.scatter(comp['bin_coarseness'], comp['gap'], s=sizes, c=comp['r2_alone'],
                cmap='viridis', edgecolor='k', linewidth=0.5)
for _, r in comp.iterrows():
    ax.annotate(r['confound'], (r['bin_coarseness'], r['gap']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('bin coarseness (100 − actual bins achieved)')
ax.set_ylabel('|stratified − model-comparison| (pct pts)')
ax.set_title('Disagreement only appears when binning is coarse\nAND the confound has real explanatory power\n(point color/size = confound-alone R²)')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('confound-alone R²')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_disagreement_vs_coarseness.pdf', bbox_inches='tight')
plt.show()